In [ ]:
!pip install -q --upgrade pip
!pip uninstall -y torch torchvision
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu118

!pip install -q --upgrade transformers accelerate sentence-transformers chromadb "pillow<10.0.0" requests qwen-vl-utils

import torch
import transformers
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info
from sentence_transformers import SentenceTransformer
import chromadb
from PIL import Image
import requests
from io import BytesIO
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient
import os
import shutil

try:
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HF_TOKEN")
    login(token=hf_token)
    print("Hugging Face login successful!")
except Exception as e:
    print(f"HuggingFace login failed (continuing if token not strictly required): {e}")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Model will run on: {device}")
INPUT_DATA_PATH = "/kaggle/input/knowledge-dataset/"

READ_ONLY_DB_PATH = os.path.join(INPUT_DATA_PATH, "embedding_vectors")
IMG_PATH = os.path.join(INPUT_DATA_PATH, "all_images")
WRITABLE_DB_PATH = "/kaggle/working/my_vector_db"

print(f"Copying database from {READ_ONLY_DB_PATH} to {WRITABLE_DB_PATH}...")

if os.path.exists(WRITABLE_DB_PATH):
    print("Writable copy already exists. Removing old one.")
    shutil.rmtree(WRITABLE_DB_PATH)

shutil.copytree(READ_ONLY_DB_PATH, WRITABLE_DB_PATH)
print("Database copied successfully.")
model_id = "Qwen/Qwen2-VL-2B-Instruct"

print("\nLoading Qwen2-VL-2B-Instruct model and processor...")
processor = AutoProcessor.from_pretrained(model_id)

model = Qwen2VLForConditionalGeneration.from_pretrained(
    model_id,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
    low_cpu_mem_usage=True
).eval()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print(f"Qwen Model loaded on: {model.device}")
print("Loading text embedding model (google/embeddinggemma-300m) on CPU...")
text_embed_model = SentenceTransformer(
    'google/embeddinggemma-300m',
    device="cpu",
    trust_remote_code=True
)
print("Embedding model loaded.")
print("Loading vector database...")
client = chromadb.PersistentClient(path=WRITABLE_DB_PATH)
collection = client.get_collection(name="assam_disasters_multimodal")
print(f"Database loaded. Total items: {collection.count()}")

def generate_qwen_response(messages, max_new_tokens=256):
    text = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt"
    )
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    with torch.no_grad():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=None,
            top_p=None,
            top_k=None
        )
    input_token_len = inputs['input_ids'].shape[1]
    output_text = processor.decode(
        generated_ids[0][input_token_len:],
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False
    )
    return output_text
def analyze_image(image_input, analysis_prompt="Describe this image in detail.", max_new_tokens=512):
    print(f"ANALYZING IMAGE: {image_input}")
    try:
        if image_input.startswith('http'):
            image = Image.open(BytesIO(requests.get(image_input).content)).convert("RGB")
        else:
            image = Image.open(image_input).convert("RGB")
        max_size = 1024
        if max(image.size) > max_size:
            ratio = max_size / max(image.size)
            new_size = tuple(int(dim * ratio) for dim in image.size)
            image = image.resize(new_size, Image.Resampling.LANCZOS)
            print(f"Resized image to: {image.size}")

    except Exception as e:
        print(f"ERROR: Could not load image from '{image_input}': {e}")
        return
    messages = [
        {
            "role": "user","content": [{"type": "image", "image": image},{"type": "text", "text": analysis_prompt}]
        }
    ]
    print("Generating analysis with Qwen...")
    answer = generate_qwen_response(messages, max_new_tokens=max_new_tokens)
    print(answer)

import time

def run_multimodal_rag(question, n_results=3, max_new_tokens=256):
    total_start = time.time()
    print(f"\n--- RAG Query ---\nQ: {question}\n")
    embed_start = time.time()
    print("Embedding question...")

    if torch.cuda.is_available():
        text_embed_model.to(device)

    query_embedding = text_embed_model.encode(question).tolist()

    text_embed_model.to("cpu")
    torch.cuda.empty_cache()
    embed_latency = time.time() - embed_start

    retrieval_start = time.time()
    print("Retrieving relevant documents...")
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=n_results
    )
    retrieval_latency = time.time() - retrieval_start

    if not results or not results['documents'][0]:
        print("Wecouldn't find any relevant documents.")
        return
    context_text_list = []
    context_image_paths = []
    documents = results['documents'][0]
    metadatas = results['metadatas'][0]

    for doc, meta in zip(documents, metadatas):
        context_text_list.append(doc)
        image_paths_str = meta.get('image_paths', '')
        if image_paths_str:
            paths = image_paths_str.split(',')
            for p in paths:
                img_filename = os.path.basename(p)
                correct_path = os.path.join(IMG_PATH, img_filename)
                if os.path.exists(correct_path):
                    context_image_paths.append(correct_path)

    context_image_paths = sorted(list(set(context_image_paths)))

    messages = [{"role": "user", "content": []}]
    MAX_IMAGES = 2
    for img_path in context_image_paths[:MAX_IMAGES]:
        try:
            rag_image = Image.open(img_path).convert("RGB")
            messages[0]["content"].append({"type": "image", "image": rag_image})
        except Exception:
            pass

    messages[0]["content"].append({"type": "text", "text": prompt})
    gen_start = time.time()
    print("Generating answer with Qwen...")
    answer = generate_qwen_response(messages, max_new_tokens=max_new_tokens)
    gen_latency = time.time() - gen_start
    total_latency = time.time() - total_start

    print("\n--- Answer ---")
    print(f"A: {answer}")
    print("\n" + "="*30)
    print("       LATENCY METRICS")
    print("="*30)
    print(f"Embedding Time:  {embed_latency:.4f} sec")
    print(f"Retrieval Time:  {retrieval_latency:.4f} sec")
    print(f"Generation Time: {gen_latency:.4f} sec")
    print(f"TOTAL TIME:      {total_latency:.4f} sec")
    print("="*30 + "\n")

def run_analysis():
    print(" Qwen2-VL Analysis")
    while True:
        choice = input("Select mode (T: Text RAG | I: Image Analysis | Q: Quit): ").upper().strip()
        if choice == 'Q':
            print("Exiting analysis.")
            break
        elif choice == 'T':
            question = input("Enter your RAG query:")
            n_results = 3
            run_multimodal_rag(question, n_results)
        elif choice == 'I':
            print("\n--- Image Analysis Mode ---")
            image_input = input("Enter Image URL or FULL local path: ")
            analysis_prompt = input("Enter the analysis prompt: ")
            analyze_image(image_input, analysis_prompt)
        else:
            print("Invalid choice. Please enter T, I, or Q.")

run_analysis()